In [0]:
%run ./config

In [0]:

from pyspark.sql.functions import col, current_timestamp

In [0]:
dbutils.widgets.dropdown(
    "trigger_type",
    "availableNow",
    ["availableNow", "once", "processingTime"]
)

trigger_type = dbutils.widgets.get("trigger_type")

print(f"Using trigger type: {trigger_type}")

In [0]:
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option(
        "cloudFiles.schemaLocation",
        ORDERS_SCHEMA_LOCATION
    )
    .option(
        "cloudFiles.schemaEvolutionMode",
        "addNewColumns"
    )
    .load(
        LANDING_ORDERS_PATH
    )
)

In [0]:
df_enriched = (
    df
    .withColumn(
        "_ingest_timestamp",
        current_timestamp()
    )
    .withColumn(
        "_source_file",
        col("_metadata.file_name")
    )
    .withColumn(
        "_source_path",
        col("_metadata.file_path")
    )
)


In [0]:
trigger_options = {
    "availableNow": {
        "availableNow": True
    },
    "once": {
        "once": True
    },
    "processingTime": {
        "processingTime": "10 seconds"
    }
}

In [0]:
BRONZE_PATH = "abfss://yanquiel@dlspl21databricks.dfs.core.windows.net/bronze/orders/"

query = (
    df_enriched.writeStream
    .format("delta")
    .option("checkpointLocation",ORDERS_CHECKPOINT_LOCATION)
    .option("mergeSchema","true")
    .trigger(**trigger_options[trigger_type])
    .start(BRONZE_PATH)
)




query.awaitTermination()

print(f"Ingestion complete into {BRONZE_PATH}")




print("Micro-batch progress:")

for batch in query.recentProgress:

    source = batch["sources"][0]

    print(
        f"batchId={batch['batchId']} "
        f"rows={source['numInputRows']} "
        f"durationMs={batch['durationMs']}"
    )


